# 운동자세 AI: 26개 운동기구 대표 운동 + 가슴 세부 7종

- **1번 셀**은 공통 엔진 설치/설정입니다. 반드시 먼저 실행하세요.
- 이후 각 운동은 **한 셀씩** 독립적으로 실행할 수 있습니다. 전체를 실행하려면 `런타임 → 모두 실행`을 선택하세요.
- 실제 GOOD/BAD 영상 라벨이 충분하면 지도학습, 그렇지 않으면 공개 시범 동작을 *임시 GOOD으로 간주한 이상탐지* 모델을 시도합니다.
- **GOOD이라고 가정한 자료는 검증된 정답이 아닙니다.** BAD 예측은 정상 시범 패턴 이탈일 뿐, 실제 부정확한 운동 자세라고 증명된 것은 아닙니다.
- 원본은 `/content/Exercise_AI_TEMP`에만 저장되고 Drive에는 학습 성공 모델 + 작은 정보 파일만 저장합니다.
- Kaggle/Roboflow는 인증이나 데이터셋 지정이 없으면 건너뛰며, Mendeley/Papers With Code는 검증된 원본 주소가 없는 운동에 대해 자동 다운로드를 주장하지 않습니다.
- BAD 영상 수동 추가: `/content/exercise_ai_input/<exercise_key>/BAD/` (코랩 세션 내 파일 업로드). GOOD도 같은 방식으로 추가할 수 있습니다.

In [ ]:
# @title
# -*- coding: utf-8 -*-
"""Exercise AI: Colab training engine. Requires user-provided labels for verified BAD."""
import os, sys, subprocess, importlib, json, re, shutil, hashlib, traceback, zipfile, time
from pathlib import Path
from collections import Counter
from datetime import datetime, timezone

def _ensure(module, package):
    try: importlib.import_module(module)
    except ImportError: subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])
for m, p in [('numpy','numpy'), ('pandas','pandas'), ('cv2','opencv-python-headless'),
             ('PIL','Pillow'), ('sklearn','scikit-learn'), ('joblib','joblib'),
             ('requests','requests'), ('mediapipe','mediapipe>=0.10.30')]:
    _ensure(m,p)
import numpy as np, pandas as pd, cv2, joblib, requests, mediapipe as mp
from PIL import Image, ImageSequence
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import classification_report, confusion_matrix
try:
    from google.colab import drive, userdata
    drive.mount('/content/drive', force_remount=False)
    DR = Path('/content/drive/MyDrive')
except ImportError:
    userdata = None
    DR = Path.cwd()

# Drive only stores trained models and small model metadata. All raw files remain in /content.
OUTPUT = DR / 'Exercise_AI_Models' / 'Equipment26'
TMP = Path('/content/Exercise_AI_TEMP') if Path('/content').exists() else Path.cwd()/'Exercise_AI_TEMP'
INPUT = Path('/content/exercise_ai_input') if Path('/content').exists() else Path.cwd()/'exercise_ai_input'
OUTPUT.mkdir(parents=True, exist_ok=True); TMP.mkdir(parents=True, exist_ok=True)
SOURCES = TMP/'source_cache'; SOURCES.mkdir(exist_ok=True)

# These are 26 equipment entries, NOT all possible exercises for every machine.
# 7 chest sub-exercises are additionally supplied after the 26 entries.
# Terms are strict enough to avoid categorizing incline as flat or dumbbell fly as barbell press.
def E(ko, cat, names, eq=None, reject=()):
    return dict(ko=ko, category=cat, names=list(names), equipment=list(eq or []), reject=list(reject))
EXERCISES = {
 'treadmill_walk':E('트레드밀 걷기','Cardio',['treadmill walking','walking treadmill'],['treadmill']),
 'recumbent_cycling':E('리컴번트 바이크','Cardio',['recumbent cycling','recumbent bike'],['recumbent bike']),
 'upright_cycling':E('업라이트 바이크','Cardio',['upright cycling','stationary cycling','exercise bike'],['stationary bike']),
 'elliptical_training':E('일립티컬','Cardio',['elliptical trainer','elliptical exercise'],['elliptical machine']),
 'seated_chest_press':E('시티드 체스트 프레스','Chest',['lever chest press','seated chest press','machine chest press'],['leverage machine','machine'],['incline','decline','fly']),
 'pec_deck_fly':E('펙덱 플라이','Chest',['lever pec deck fly','pec deck fly','machine fly','butterfly'],['leverage machine','machine'],['reverse','rear delt']),
 'high_fly':E('하이 플라이','Chest',['high cable fly','high to low cable fly','cable cross over'],['cable'],['low to high']),
 'machine_shoulder_press':E('머신 숄더 프레스','Shoulder',['lever shoulder press','machine shoulder press','seated shoulder press'],['leverage machine','machine'],['dumbbell','barbell']),
 'seated_cable_row':E('롱 풀 시티드 로우','Back',['cable seated row','seated cable row','cable low row'],['cable'],['upright']),
 'machine_biceps_curl':E('머신 암 컬','Arms',['lever preacher curl','lever biceps curl','machine biceps curl'],['leverage machine','machine'],['dumbbell','barbell']),
 'abdominal_crunch':E('앱도미널 크런치','Core',['lever seated crunch','machine abdominal crunch','abdominal crunch'],['leverage machine','machine']),
 'chin_up':E('친업','Back',['chin up','chin-up'],['body weight'],['assisted dip']),
 'rotary_torso':E('로터리 토르소','Core',['lever seated twist','rotary torso','torso rotation'],['leverage machine','machine']),
 'leg_extension':E('레그 익스텐션','Legs',['lever leg extension','machine leg extension','leg extension'],['leverage machine','machine'],['curl']),
 'leg_curl':E('레그 컬','Legs',['lever seated leg curl','lever lying leg curl','leg curl'],['leverage machine','machine'],['extension']),
 'seated_leg_press':E('시티드 레그 프레스','Legs',['lever seated leg press','seated leg press','horizontal leg press'],['leverage machine','machine']),
 'cable_crossover':E('케이블 크로스오버','Chest',['cable cross over','cable crossover','cable standing fly'],['cable'],['reverse']),
 'smith_squat':E('스미스 스쿼트','Legs',['smith squat','smith machine squat','smith back squat'],['smith machine']),
 'mega_max_lat_pulldown':E('메가맥스 랫 풀다운','Back',['cable lat pulldown','lat pulldown'],['cable'],['underhand curl']),
 'dip_station_dip':E('치닝 디핑 머신 딥스','Chest',['chest dip','triceps dip','parallel bar dip'],['body weight'],['bench dip']),
 'power_leg_press':E('파워 레그 프레스','Legs',['sled 45 leg press','45 degree leg press','sled leg press'],['sled machine','leverage machine']),
 'power_rack_squat':E('파워랙 바벨 스쿼트','Legs',['barbell full squat','barbell squat','barbell back squat'],['barbell'],['smith','front']),
 'sit_up':E('싯업','Core',['sit up','sit-up','decline sit up'],['body weight'],['twist']),
 'double_twist':E('더블 트위스트','Core',['standing waist twist','standing twist','waist twist'],['body weight','machine']),
 'hyperextension':E('하이퍼 익스텐션','Back',['hyperextension','back extension','45 degree back extension'],['body weight','leverage machine'],['reverse hyper']),
 'flat_bench_press':E('슈퍼 벤치 프레스','Chest',['barbell bench press','barbell flat bench press','flat bench press'],['barbell'],['incline','decline','dumbbell','smith']),
 # Additional detailed chest movements beyond 26 representative equipment mappings.
 'incline_bench_press':E('인클라인 벤치프레스','Chest',['barbell incline bench press','incline barbell bench press','incline bench press'],['barbell'],['dumbbell','machine','smith','fly']),
 'decline_bench_press':E('디클라인 벤치프레스','Chest',['barbell decline bench press','decline barbell bench press','decline bench press'],['barbell'],['dumbbell','machine','smith','fly']),
 'machine_chest_press':E('머신 체스트 프레스 (가슴 세부)','Chest',['lever chest press','seated chest press','machine chest press'],['leverage machine','machine'],['incline','decline','fly']),
 'chest_fly':E('체스트 플라이','Chest',['pec deck fly','machine fly','cable chest fly'],['cable','leverage machine','machine'],['incline','reverse','high to low','low to high']),
 'high_to_low_fly':E('하이 투 로우 플라이','Chest',['high to low cable fly','high cable fly'],['cable'],['low to high']),
 'low_to_high_fly':E('로우 투 하이 플라이','Chest',['low to high cable fly','low cable fly'],['cable'],['high to low']),
 'incline_dumbbell_fly':E('인클라인 덤벨 플라이','Chest',['incline dumbbell fly','incline dumbbell chest fly'],['dumbbell'],['press','decline','flat'])
}
EQUIPMENT_26_KEYS = list(EXERCISES)[:26]
CHEST_EXTRA_KEYS = list(EXERCISES)[26:]
assert len(EQUIPMENT_26_KEYS)==26 and len(CHEST_EXTRA_KEYS)==7

MAX_GITHUB_RECORDS = 28
MAX_FRAMES_VIDEO = 16
MIN_GOOD_FILES = 3
MIN_GOOD_POSES = 12
MAX_SOURCE_MEDIA = 90
MAX_MEDIA_MB = 220
# Pre-approved known projects, editable. Do NOT treat them as accuracy-labelled.
ROBOFLOW_PROJECTS = {
    'flat_bench_press': [('universityworkdata','bench-press-g4vqm',1),('thomas-bardhi-d9m1v','bench-press-detection',2)]
}
# Optional datasets: annotate suitable dataset handles (requires Kaggle authentication).
KAGGLE_DATASETS = {'flat_bench_press':['hasyimabdillah/workoutfitness-video']}


def secret(key):
    val = os.environ.get(key,'')
    if val: return val
    if userdata:
        try: return userdata.get(key) or ''
        except Exception: pass
    return ''

def norm(s): return ' '.join(re.sub(r'[^a-z0-9]+',' ',str(s).lower()).split())

def accepted(name,spec,eq=''):
    n = norm(name)
    if any(norm(w) in n for w in spec['reject']): return False
    if not any(norm(w) in n for w in spec['names']): return False
    # Avoid accidental substitution of barbell vs smith or generic machine.
    if spec['equipment'] and eq:
        eq = norm(eq)
        return any(norm(z) in eq or eq in norm(z) for z in spec['equipment'])
    return True

session = requests.Session()
session.headers.update({'User-Agent':'ExerciseAIColabResearch/1.0'})

def load_json_sources():
    resources = [
      ('blackjyn','https://raw.githubusercontent.com/blackjyn/gym-exercises-dataset/main/data/exercises.json',
       'https://raw.githubusercontent.com/blackjyn/gym-exercises-dataset/main/'),
      ('yuhonas','https://raw.githubusercontent.com/yuhonas/free-exercise-db/main/dist/exercises.json',
       'https://raw.githubusercontent.com/yuhonas/free-exercise-db/main/exercises/')]
    result=[]
    for name,url,prefix in resources:
        try:
            r=session.get(url,timeout=45);r.raise_for_status()
            obj=r.json()
            items=obj if isinstance(obj,list) else obj.get('exercises',obj.get('data',[]))
            if not isinstance(items,list): raise ValueError('metadata JSON list not found')
            result.append((name,items,prefix))
            print(f'[데이터 색인] {name}: {len(items)}개 동작')
        except Exception as e: print(f'[데이터 색인] {name}: 실패 {type(e).__name__}: {str(e)[:120]}')
    return result
GITHUB_INDEX = load_json_sources()

# MediaPipe Tasks uses current API compatible with Python 3.13.
POSE_MODEL_URL='https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_lite/float16/1/pose_landmarker_lite.task'
POSE_ASSET=TMP/'pose_landmarker_lite.task'

def prepare_pose_model():
    if POSE_ASSET.exists() and POSE_ASSET.stat().st_size>1000000: return
    r=session.get(POSE_MODEL_URL,timeout=120); r.raise_for_status();POSE_ASSET.write_bytes(r.content)

def safe_download(url,dest,max_bytes=25_000_000):
    if dest.exists() and dest.stat().st_size>1024:return True
    if not url.startswith('https://'): return False
    try:
        with session.get(url,timeout=35,stream=True) as r:
            r.raise_for_status(); total=0
            with dest.open('wb') as f:
                for chunk in r.iter_content(256*1024):
                    if chunk:
                        total+=len(chunk)
                        if total>max_bytes: raise ValueError('file too large')
                        f.write(chunk)
        if dest.stat().st_size<=1024: raise ValueError('file empty')
        return True
    except Exception:
        dest.unlink(missing_ok=True);return False

def collect_github(key,spec,folder,events):
    downloaded=0; hits=0
    for db,items,root in GITHUB_INDEX:
        match=[]
        for item in items:
            if isinstance(item,dict) and accepted(item.get('name',''),spec,item.get('equipment','')):
                match.append(item)
        hits+=len(match)
        for entry in match[:MAX_GITHUB_RECORDS]:
            media=[]
            if db=='blackjyn':
                for field in ('image','gif_url'):
                    val=entry.get(field,'')
                    if isinstance(val,dict):val=val.get('url','')
                    if isinstance(val,str) and val:media.append(val)
            else:
                media=[m for m in entry.get('images',[]) if isinstance(m,str)]
            for m in media[:2]:
                url = m if m.startswith('https://') else root+m.lstrip('/')
                extension = Path(m.split('?')[0]).suffix.lower()
                if extension not in ('.jpg','.jpeg','.png','.gif','.webp'): continue
                filename=f'{db}_{hashlib.sha1(url.encode()).hexdigest()[:16]}{extension}'
                if safe_download(url,folder/filename): downloaded+=1
                if downloaded>=MAX_SOURCE_MEDIA:break
            if downloaded>=MAX_SOURCE_MEDIA:break
    events.append({'site':'GitHub','status':'downloaded' if downloaded else 'no_download','matched_records':hits,'media':downloaded,
                   'label':'ASSUMED_GOOD_DEMONSTRATION_NOT_VERIFIED'})

def collect_kaggle(key,spec,folder,events):
    handles=KAGGLE_DATASETS.get(key,[])
    if not handles:
        events.append({'site':'Kaggle','status':'not_configured','note':'운동별 데이터셋 확정 주소가 없어 자동 수집하지 않음'})
        return
    username,password=secret('KAGGLE_USERNAME'),secret('KAGGLE_KEY')
    if not (username and password):
        events.append({'site':'Kaggle','status':'authentication_required'})
        return
    os.environ['KAGGLE_USERNAME']=username;os.environ['KAGGLE_KEY']=password
    try:
        _ensure('kaggle','kaggle')
        from kaggle.api.kaggle_api_extended import KaggleApi
        api=KaggleApi();api.authenticate()
    except BaseException as e:
        events.append({'site':'Kaggle','status':'auth_error','message':str(e)[:250]});return
    files_saved=0
    for handle in handles:
        try:
            entries=api.dataset_list_files(handle).files or []
            matches=[x.name for x in entries if accepted(x.name,spec) and Path(x.name).suffix.lower() in ('.mp4','.mov','.avi','.mkv','.jpg','.jpeg','.png')]
            for filename in matches[:MAX_SOURCE_MEDIA]:
                sd=folder/('kg_'+hashlib.sha1((handle+filename).encode()).hexdigest()[:12]);sd.mkdir(exist_ok=True)
                try:
                    api.dataset_download_file(handle,filename,path=str(sd),quiet=True)
                    for archive in sd.glob('*.zip'):
                        with zipfile.ZipFile(archive) as z:
                            for f in z.infolist():
                                if Path(f.filename).suffix.lower() not in ('.mp4','.mov','.avi','.mkv','.jpg','.jpeg','.png'):continue
                                if f.file_size>MAX_MEDIA_MB*1024*1024:continue
                                path=sd/Path(f.filename).name
                                with z.open(f) as fi, path.open('wb') as fo:shutil.copyfileobj(fi,fo)
                        archive.unlink(missing_ok=True)
                    files_saved+=1
                except Exception:pass
        except Exception as e:events.append({'site':'Kaggle','status':'dataset_error','handle':handle,'error':str(e)[:150]})
    events.append({'site':'Kaggle','status':'attempted','files_downloaded':files_saved})

def collect_roboflow(key,folder,events):
    projects=ROBOFLOW_PROJECTS.get(key,[])
    if not projects:
        events.append({'site':'Roboflow','status':'not_configured'});return
    api_key=secret('ROBOFLOW_API_KEY')
    if not api_key:
        events.append({'site':'Roboflow','status':'authentication_required'});return
    try:
        _ensure('roboflow','roboflow')
        from roboflow import Roboflow
        rf=Roboflow(api_key=api_key)
    except Exception as e:
        events.append({'site':'Roboflow','status':'setup_error','error':str(e)[:150]});return
    for ws,proj,ver in projects:
        try:
            rf.workspace(ws).project(proj).version(ver).download('yolov8',location=str(folder/f'roboflow_{proj}_{ver}'))
            events.append({'site':'Roboflow','status':'downloaded','dataset':f'{ws}/{proj}/{ver}'})
        except Exception as e:events.append({'site':'Roboflow','status':'access_error','dataset':proj,'error':str(e)[:150]})

def iter_frames(path):
    ext=path.suffix.lower()
    if ext in ('.jpg','.jpeg','.png','.webp'):
        im=cv2.imread(str(path))
        if im is not None:yield im
    elif ext=='.gif':
        try:
            with Image.open(path) as im:
                n=getattr(im,'n_frames',1)
                for ind in sorted(set([0,n//3,2*n//3])):
                    im.seek(ind)
                    rgb=np.array(im.convert('RGB'))
                    yield cv2.cvtColor(rgb,cv2.COLOR_RGB2BGR)
        except Exception: return
    elif ext in ('.mp4','.avi','.mov','.mkv','.webm'):
        cap=cv2.VideoCapture(str(path))
        try:
            total=int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
            if total<2:return
            for j in sorted(set(np.linspace(0,total-1,MAX_FRAMES_VIDEO).astype(int))):
                cap.set(cv2.CAP_PROP_POS_FRAMES,int(j));ok,frame=cap.read()
                if ok:yield frame
        finally:cap.release()

def pose_feature(frame,detector):
    if frame is None:return None
    h,w=frame.shape[:2]
    if min(h,w)<32:return None
    scale=min(1.,640/max(h,w))
    if scale<1:frame=cv2.resize(frame,(max(1,int(w*scale)),max(1,int(h*scale))))
    image=np.ascontiguousarray(cv2.cvtColor(frame,cv2.COLOR_BGR2RGB))
    out=detector.detect(mp.Image(image_format=mp.ImageFormat.SRGB,data=image))
    if not out.pose_landmarks:return None
    lm=out.pose_landmarks[0]
    ids=[11,12,13,14,15,16,23,24,25,26,27,28]
    mat=np.array([[lm[i].x,lm[i].y] for i in ids],dtype=np.float32)
    if not np.isfinite(mat).all():return None
    midpoint=(mat[0]+mat[1])/2
    hips=(mat[6]+mat[7])/2
    sz=max(float(np.linalg.norm(midpoint-hips)),float(np.linalg.norm(mat[0]-mat[1])),0.08)
    normalized=(mat-midpoint)/sz
    return normalized.flatten().astype(np.float32)

def gather_files(key,folder):
    items=[]
    media_ext={'.jpg','.jpeg','.png','.webp','.gif','.mp4','.mov','.avi','.mkv','.webm'}
    for file in folder.rglob('*'):
        if file.is_file() and file.suffix.lower() in media_ext:
            # Roboflow label overlays or generated test previews are not treated as source photos
            if 'roboflow_' in str(file) and 'images' not in file.parts:continue
            items.append((file,'GOOD','auto_assumed_demo'))
    for label in ['GOOD','BAD']:
        manual=INPUT/key/label
        if manual.exists():
            for file in manual.rglob('*'):
                if file.is_file() and file.suffix.lower() in media_ext:
                    items.append((file,label,'user_label'))
    return items

RESULTS=[]
def train_one(key):
    if key not in EXERCISES: raise KeyError(f'등록되지 않은 운동: {key}')
    spec=EXERCISES[key]; print('\n'+'='*65+'\n'+f'{spec["ko"]} [{key}]')
    folder=TMP/key;folder.mkdir(parents=True,exist_ok=True)
    ev=[];collect_github(key,spec,folder,ev);collect_kaggle(key,spec,folder,ev);collect_roboflow(key,folder,ev)
    ev.append({'site':'Mendeley','status':'index_only','note':'공개 색인; 이 운동의 검증된 원본 다운로드 주소 미설정'})
    ev.append({'site':'Papers With Code','status':'index_only','note':'데이터셋 색인; 이 운동의 검증된 원본 다운로드 주소 미설정'})
    files=gather_files(key,folder); unique_media=len(set(str(p) for p,_,_ in files))
    print('후보 미디어:',unique_media)
    report={'exercise':key,'name_ko':spec['ko'],'equipment_group':spec['category'],
            'unique_media_files':unique_media,'pose_samples':0,'status':'NOT_TRAINED',
            'model_path':None,'sources':ev,'training_mode':None,
            'warning':'시범 이미지/영상은 요청에 따라 GOOD으로 가정. 실제 올바른 자세 검증이 아님.'}
    if not files:
        report['status']='NOT_TRAINED_NO_DATA';RESULTS.append(report);print(report['status']);return report
    try:prepare_pose_model()
    except Exception as e:
        report.update(status='NOT_TRAINED_POSE_DOWNLOAD_ERROR',error=str(e)[:250]);RESULTS.append(report);return report
    X=[];Y=[];groups=[];good_files=set(); errors=[]; seen=set()
    settings=mp.tasks.vision.PoseLandmarkerOptions(
        base_options=mp.tasks.BaseOptions(model_asset_path=str(POSE_ASSET)),
        running_mode=mp.tasks.vision.RunningMode.IMAGE, num_poses=1,
        min_pose_detection_confidence=.45,min_pose_presence_confidence=.45)
    try:
        with mp.tasks.vision.PoseLandmarker.create_from_options(settings) as detector:
            for idx,(path,label,origin) in enumerate(files,1):
                if idx>MAX_SOURCE_MEDIA+200:break
                try:
                    digest=hashlib.sha256(path.read_bytes()).hexdigest()
                    if digest in seen:continue
                    seen.add(digest)
                    file_has_pose=False
                    for frame in iter_frames(path):
                        feat=pose_feature(frame,detector)
                        if feat is not None:
                            X.append(feat);Y.append(label);groups.append(digest)
                            file_has_pose=True
                    if label=='GOOD' and file_has_pose:good_files.add(digest)
                except Exception as e:errors.append(str(e)[:150])
    except Exception as e:
        report.update(status='NOT_TRAINED_POSE_ERROR',error=str(e)[:250]);RESULTS.append(report);return report
    report['pose_samples']=len(X);report['unique_good_pose_files']=len(good_files)
    if errors:report['sample_errors']=errors[:3]
    if len(good_files)<MIN_GOOD_FILES or sum(z=='GOOD' for z in Y)<MIN_GOOD_POSES:
        report['status']='NOT_TRAINED_INSUFFICIENT_POSES';RESULTS.append(report)
        print('학습 미진행: GOOD 포즈가 추출된 서로 다른 파일 부족',len(good_files),' / 총 GOOD 포즈',sum(z=='GOOD' for z in Y))
        return report
    X=np.asarray(X,dtype=np.float32);Y=np.asarray(Y);groups=np.asarray(groups)
    good=X[Y=='GOOD']
    normal_model=make_pipeline(StandardScaler(),IsolationForest(n_estimators=220,contamination=.08,random_state=42,n_jobs=-1))
    normal_model.fit(good)
    threshold=float(np.quantile(normal_model.score_samples(good),.08))
    model_obj={'model':normal_model,'threshold':threshold,'exercise':key,'label_mode':'GOOD_ONLY_ANOMALY',
               'feature_version':'pose12_shoulder_normalized_v1','pose_asset_url':POSE_MODEL_URL,
               'feature_size':int(X.shape[1]),'assumed_good_sources':True,
               'warning':'BAD 의미는 정상 시범 패턴과 차이남. 실제 부상 위험 및 오류의 검증된 예측 아님.'}
    mode='TRAINED_GOOD_ONLY'
    # If labelled BAD clips exist, build a distinct supervised classifier, subject to group separation.
    if len(set(Y))==2 and len(set(groups[Y=='GOOD']))>=3 and len(set(groups[Y=='BAD']))>=3 and sum(Y=='BAD')>=12:
        split=GroupShuffleSplit(n_splits=60,test_size=.25,random_state=42)
        for tr,te in split.split(X,Y,groups):
            if set(Y[tr])=={'GOOD','BAD'} and set(Y[te])=={'GOOD','BAD'}:
                clf=RandomForestClassifier(n_estimators=240,class_weight='balanced_subsample',random_state=42,n_jobs=-1)
                clf.fit(X[tr],Y[tr]); preds=clf.predict(X[te])
                model_obj['supervised_model']=clf
                model_obj['evaluation']=classification_report(Y[te],preds,output_dict=True,zero_division=0)
                mode='TRAINED_GOOD_BAD'; break
    model_obj['label_mode']=mode
    dest=OUTPUT/spec['category'];dest.mkdir(parents=True,exist_ok=True)
    model_file=dest/f'{key}.joblib';joblib.dump(model_obj,model_file)
    report.update(status=mode,model_path=str(model_file),training_mode=mode)
    info_file=dest/f'{key}_info.json'
    # source descriptions are small provenance metadata only; no raw images/videos are written to Drive.
    info_file.write_text(json.dumps(report,ensure_ascii=False,indent=2),encoding='utf-8')
    RESULTS.append(report)
    print('학습 결과:',mode,'| 관절 샘플:',len(X),'| 모델:',model_file)
    return report

def train_safe(key):
    try:return train_one(key)
    except Exception as e:
        print('운동별 실패:',key,type(e).__name__,str(e)[:250])
        result={'exercise':key,'status':'ERROR','error':str(e)[:250],'unique_media_files':0,'pose_samples':0,'model_path':None}
        RESULTS.append(result);return result

def show_summary():
    cols=['exercise','status','unique_media_files','pose_samples','model_path']
    df=pd.DataFrame([{k:r.get(k) for k in cols} for r in RESULTS]);display(df)
    print('총 처리:',len(df),'| 학습 성공:',sum(str(x).startswith('TRAINED') for x in df.status),'| 실패/자료부족:',sum(not str(x).startswith('TRAINED') for x in df.status))
    return df

def cleanup_temporary():
    shutil.rmtree(TMP,ignore_errors=True)
    print('코랩 임시 원본 삭제 완료. Drive의 모델은 유지됩니다.')

print('\n준비 완료: 기구 대표 운동',len(EQUIPMENT_26_KEYS),'개 + 가슴 세부 운동',len(CHEST_EXTRA_KEYS),'개')
print('원본 임시 위치:',TMP,' / 최종 모델:',OUTPUT)


### 기구 대표 운동 01/26 — `treadmill_walk`

In [ ]:
# @title
result_01 = train_safe('treadmill_walk')
print({k:result_01.get(k) for k in ('exercise','status','unique_media_files','pose_samples','model_path')})

### 기구 대표 운동 02/26 — `recumbent_cycling`

In [ ]:
result_02 = train_safe('recumbent_cycling')
print({k:result_02.get(k) for k in ('exercise','status','unique_media_files','pose_samples','model_path')})

### 기구 대표 운동 03/26 — `upright_cycling`

In [ ]:
result_03 = train_safe('upright_cycling')
print({k:result_03.get(k) for k in ('exercise','status','unique_media_files','pose_samples','model_path')})

### 기구 대표 운동 04/26 — `elliptical_training`

In [ ]:
result_04 = train_safe('elliptical_training')
print({k:result_04.get(k) for k in ('exercise','status','unique_media_files','pose_samples','model_path')})

### 기구 대표 운동 05/26 — `seated_chest_press`

In [ ]:
result_05 = train_safe('seated_chest_press')
print({k:result_05.get(k) for k in ('exercise','status','unique_media_files','pose_samples','model_path')})

### 기구 대표 운동 06/26 — `pec_deck_fly`

In [ ]:
result_06 = train_safe('pec_deck_fly')
print({k:result_06.get(k) for k in ('exercise','status','unique_media_files','pose_samples','model_path')})

### 기구 대표 운동 07/26 — `high_fly`

In [ ]:
result_07 = train_safe('high_fly')
print({k:result_07.get(k) for k in ('exercise','status','unique_media_files','pose_samples','model_path')})

### 기구 대표 운동 08/26 — `machine_shoulder_press`

In [ ]:
result_08 = train_safe('machine_shoulder_press')
print({k:result_08.get(k) for k in ('exercise','status','unique_media_files','pose_samples','model_path')})

### 기구 대표 운동 09/26 — `seated_cable_row`

In [ ]:
result_09 = train_safe('seated_cable_row')
print({k:result_09.get(k) for k in ('exercise','status','unique_media_files','pose_samples','model_path')})

### 기구 대표 운동 10/26 — `machine_biceps_curl`

In [ ]:
result_10 = train_safe('machine_biceps_curl')
print({k:result_10.get(k) for k in ('exercise','status','unique_media_files','pose_samples','model_path')})

### 기구 대표 운동 11/26 — `abdominal_crunch`

In [ ]:
result_11 = train_safe('abdominal_crunch')
print({k:result_11.get(k) for k in ('exercise','status','unique_media_files','pose_samples','model_path')})

### 기구 대표 운동 12/26 — `chin_up`

In [ ]:
result_12 = train_safe('chin_up')
print({k:result_12.get(k) for k in ('exercise','status','unique_media_files','pose_samples','model_path')})

### 기구 대표 운동 13/26 — `rotary_torso`

In [ ]:
result_13 = train_safe('rotary_torso')
print({k:result_13.get(k) for k in ('exercise','status','unique_media_files','pose_samples','model_path')})

### 기구 대표 운동 14/26 — `leg_extension`

In [ ]:
result_14 = train_safe('leg_extension')
print({k:result_14.get(k) for k in ('exercise','status','unique_media_files','pose_samples','model_path')})

### 기구 대표 운동 15/26 — `leg_curl`

In [ ]:
result_15 = train_safe('leg_curl')
print({k:result_15.get(k) for k in ('exercise','status','unique_media_files','pose_samples','model_path')})

### 기구 대표 운동 16/26 — `seated_leg_press`

In [ ]:
result_16 = train_safe('seated_leg_press')
print({k:result_16.get(k) for k in ('exercise','status','unique_media_files','pose_samples','model_path')})

### 기구 대표 운동 17/26 — `cable_crossover`

In [ ]:
result_17 = train_safe('cable_crossover')
print({k:result_17.get(k) for k in ('exercise','status','unique_media_files','pose_samples','model_path')})

### 기구 대표 운동 18/26 — `smith_squat`

In [ ]:
result_18 = train_safe('smith_squat')
print({k:result_18.get(k) for k in ('exercise','status','unique_media_files','pose_samples','model_path')})

### 기구 대표 운동 19/26 — `mega_max_lat_pulldown`

In [ ]:
result_19 = train_safe('mega_max_lat_pulldown')
print({k:result_19.get(k) for k in ('exercise','status','unique_media_files','pose_samples','model_path')})

### 기구 대표 운동 20/26 — `dip_station_dip`

In [ ]:
result_20 = train_safe('dip_station_dip')
print({k:result_20.get(k) for k in ('exercise','status','unique_media_files','pose_samples','model_path')})

### 기구 대표 운동 21/26 — `power_leg_press`

In [ ]:
result_21 = train_safe('power_leg_press')
print({k:result_21.get(k) for k in ('exercise','status','unique_media_files','pose_samples','model_path')})

### 기구 대표 운동 22/26 — `power_rack_squat`

In [ ]:
result_22 = train_safe('power_rack_squat')
print({k:result_22.get(k) for k in ('exercise','status','unique_media_files','pose_samples','model_path')})

### 기구 대표 운동 23/26 — `sit_up`

In [ ]:
result_23 = train_safe('sit_up')
print({k:result_23.get(k) for k in ('exercise','status','unique_media_files','pose_samples','model_path')})

### 기구 대표 운동 24/26 — `double_twist`

In [ ]:
result_24 = train_safe('double_twist')
print({k:result_24.get(k) for k in ('exercise','status','unique_media_files','pose_samples','model_path')})

### 기구 대표 운동 25/26 — `hyperextension`

In [ ]:
result_25 = train_safe('hyperextension')
print({k:result_25.get(k) for k in ('exercise','status','unique_media_files','pose_samples','model_path')})

### 기구 대표 운동 26/26 — `flat_bench_press`

In [ ]:
result_26 = train_safe('flat_bench_press')
print({k:result_26.get(k) for k in ('exercise','status','unique_media_files','pose_samples','model_path')})

### 가슴 세부 추가 01/7 — `incline_bench_press`

In [ ]:
chest_01 = train_safe('incline_bench_press')
print({k:chest_01.get(k) for k in ('exercise','status','unique_media_files','pose_samples','model_path')})

### 가슴 세부 추가 02/7 — `decline_bench_press`

In [ ]:
chest_02 = train_safe('decline_bench_press')
print({k:chest_02.get(k) for k in ('exercise','status','unique_media_files','pose_samples','model_path')})

### 가슴 세부 추가 03/7 — `machine_chest_press`

In [ ]:
chest_03 = train_safe('machine_chest_press')
print({k:chest_03.get(k) for k in ('exercise','status','unique_media_files','pose_samples','model_path')})

### 가슴 세부 추가 04/7 — `chest_fly`

In [ ]:
chest_04 = train_safe('chest_fly')
print({k:chest_04.get(k) for k in ('exercise','status','unique_media_files','pose_samples','model_path')})

### 가슴 세부 추가 05/7 — `high_to_low_fly`

In [ ]:
chest_05 = train_safe('high_to_low_fly')
print({k:chest_05.get(k) for k in ('exercise','status','unique_media_files','pose_samples','model_path')})

### 가슴 세부 추가 06/7 — `low_to_high_fly`

In [ ]:
chest_06 = train_safe('low_to_high_fly')
print({k:chest_06.get(k) for k in ('exercise','status','unique_media_files','pose_samples','model_path')})

### 가슴 세부 추가 07/7 — `incline_dumbbell_fly`

In [ ]:
chest_07 = train_safe('incline_dumbbell_fly')
print({k:chest_07.get(k) for k in ('exercise','status','unique_media_files','pose_samples','model_path')})

### 학습 상태 일괄 확인

In [ ]:
summary = show_summary()

### 선택: 코랩의 임시 다운로드/전처리 파일 삭제 (Drive 모델 유지)
기록을 확인한 뒤 이 셀을 실행하세요.

In [ ]:
cleanup_temporary()